In [1]:
#!/usr/bin/env python3"""DeBERTa-v3 + Keystroke Feature Injection for Holistic Essay Score Prediction.Approach: Concatenate keystroke feature vector with DeBERTa's [CLS] embeddingbefore the regression head:  [CLS] ⊕ keystroke_features → MLP → scoreMinimal adaptation of the text-only baseline — uses AutoModelForSequenceClassificationas the base and only replaces the classifier head, keeping all HF Trainer internals intact."""

In [2]:
import argparse
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 512

SPLITS_DIR = Path("BEA Paper/combined")
TEXT_LOOKUP = Path("BEA Paper/text_mega_lookup.tsv")
OUTPUT_DIR = Path("BEA Paper/models_transformer/text_plus_features")

SCORE_COL = "holistic_score"

# Keystroke feature columns (already z-scored per source_timepoint in the CSVs)
FEATURE_COLS = [
    "total_writing_time", "initial_pause",
    "break_count", "break_total_time", "break_mean_duration", "break_ratio",
    "burst_count", "burst_mean_length_char", "burst_mean_duration",
    "deletion_count", "deletion_ratio", "deletion_char_count",
    "total_keystrokes", "final_text_length_char", "final_text_length_word",
    "chars_per_minute", "process_product_ratio",
    "navigation_count", "copy_paste_count", "linearity_index",
    "time_in_text_ms", "time_in_task_text_ms",
    "time_in_chat_ms", "time_in_chat_prompt_ms", "time_in_none_ms",
    "target_switches",
]

# Source timepoints (used for one-hot encoding so the model knows
# which normalisation group the z-scored features belong to)
TIMEPOINTS = ["05min", "10min", "15min", "20min", "25min", "full"]

# Training hyperparameters
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 10
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42

# Feature injection MLP settings
FEATURE_MLP_HIDDEN = 128
FEATURE_DROPOUT = 0.1

# Early stopping
EARLY_STOPPING_PATIENCE = 3


In [4]:
def load_text_lookup(path: Path) -> dict:
    """
    Load text lookup TSV.

    Supports both formats:
      4-col (original):  filename, task, chat_used, text
      5-col (hybrid):    filename, task, chat_used, source_timepoint, text

    Returns dict keyed on (filename, source_timepoint) for the hybrid format,
    or (filename, 'full') for the original format.
    """
    lookup = {}
    with open(path, 'r', encoding='utf-8') as f:
        header = f.readline().strip()
        n_cols = len(header.split('\t'))

        for line in f:
            parts = line.rstrip('\n').split('\t', n_cols - 1)

            if n_cols == 5 and len(parts) == 5:
                filename, task, chat_used, source_timepoint, text_escaped = parts
            elif n_cols == 4 and len(parts) == 4:
                filename, task, chat_used, text_escaped = parts
                source_timepoint = 'full'
            else:
                continue

            text = (
                text_escaped.replace('\\n', '\n')
                           .replace('\\r', '\r')
                           .replace('\\t', '\t')
                           .replace('\\\\', '\\')
            )
            lookup[(filename, source_timepoint)] = text
    return lookup

def normalise_filename(fn: str) -> str:
    """Normalise filenames for matching between splits and text lookup."""
    return fn.strip()


def timepoint_one_hot(tp: str, timepoints: list) -> list:
    """One-hot encode a source_timepoint string."""
    vec = [0.0] * len(timepoints)
    if tp in timepoints:
        vec[timepoints.index(tp)] = 1.0
    return vec


# ============================================================
# DATASET  (text + z-scored keystroke features + timepoint)
# ============================================================

class EssayWithFeaturesDataset(Dataset):
    """
    Dataset for essay scoring: text + keystroke features -> score.

    Features are already z-scored per source_timepoint in the CSVs,
    so we do NOT re-normalise here.  Instead we append a one-hot
    encoding of source_timepoint so the model knows which
    normalisation group each sample belongs to.
    """

    def __init__(self, df: pd.DataFrame, text_lookup: dict, tokenizer,
                    max_length: int, feature_cols: list, timepoints: list):
            self.tokenizer = tokenizer
            self.max_length = max_length
            self.num_keystroke_features = len(feature_cols)
            self.num_timepoint_features = len(timepoints)
            # Total feature vector: keystroke features + timepoint one-hot
            self.num_features = self.num_keystroke_features + self.num_timepoint_features

            self.texts = []
            self.features = []
            self.scores = []
            self.filenames = []
            self.chat_used = []
            self.timepoints = []

            missing = 0
            for _, row in df.iterrows():
                fn = normalise_filename(row['filename'])
                source_tp = row.get('_source_timepoint', row.get('source_timepoint', 'full'))

                text = text_lookup.get((fn, source_tp))
                if text is None:
                    text = text_lookup.get((fn, 'full'))
                if text is None:
                    missing += 1
                    continue

                # Z-scored keystroke features (as-is from CSV)
                feat_values = []
                for col in feature_cols:
                    val = row.get(col, 0.0)
                    feat_values.append(float(val) if pd.notna(val) else 0.0)

                # Append one-hot timepoint encoding
                tp_vec = timepoint_one_hot(source_tp, timepoints)
                feat_values.extend(tp_vec)

                self.texts.append(text)
                self.features.append(feat_values)
                self.scores.append(float(row[SCORE_COL]))
                self.filenames.append(fn)
                self.timepoints.append(source_tp)

                chat_val = row.get('chat_used', None)
                if chat_val is None:
                    chat_val = 'unknown'
                elif isinstance(chat_val, bool) or (isinstance(chat_val, str) and chat_val.lower() in ('true', 'false')):
                    chat_val = 'withchat' if str(chat_val).lower() == 'true' else 'wo_chat'
                self.chat_used.append(chat_val)

            if missing > 0:
                print(f"    Warning: {missing}/{len(df)} texts not found in lookup")

            self.features = np.array(self.features, dtype=np.float32)

            # Clip extreme z-scores — values beyond ±5 are outliers from
            # small-group normalisation and destabilise training
            n_clipped = (np.abs(self.features[:, :len(feature_cols)]) > 5.0).sum()
            self.features[:, :len(feature_cols)] = np.clip(
                self.features[:, :len(feature_cols)], -5.0, 5.0
            )
            if n_clipped > 0:
                print(f"    Clipped {n_clipped} extreme z-score values to ±5.0")

            print(f"    Loaded {len(self.texts)} samples — "
                  f"{self.num_keystroke_features} keystroke features + "
                  f"{self.num_timepoint_features} timepoint indicators = "
                  f"{self.num_features} total features")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'keystroke_features': torch.tensor(self.features[idx], dtype=torch.float32),
            'labels': torch.tensor(self.scores[idx], dtype=torch.float32),
        }


In [5]:
# ============================================================
# EVALUATION
# ============================================================

def bin_to_half(values):
    """Round to nearest 0.5."""
    return (np.array(values) * 2).round() / 2


def compute_metrics_from_predictions(y_true, y_pred) -> dict:
    """Compute all evaluation metrics."""
    pearson_r, pearson_p = stats.pearsonr(y_true, y_pred)
    spearman_r, spearman_p = stats.spearmanr(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # QWK with binning to 0.5 steps
    y_true_binned = bin_to_half(y_true)
    y_pred_clipped = np.clip(y_pred, 0.0, 5.0)
    y_pred_binned = bin_to_half(y_pred_clipped)

    all_labels = [f"{x:.1f}" for x in np.arange(0, 5.5, 0.5)]
    y_true_str = [f"{x:.1f}" for x in y_true_binned]
    y_pred_str = [f"{x:.1f}" for x in y_pred_binned]
    qwk = cohen_kappa_score(y_true_str, y_pred_str, weights='quadratic', labels=all_labels)

    return {
        'Pearson r': pearson_r,
        'Pearson p': pearson_p,
        'Spearman ρ': spearman_r,
        'Spearman p': spearman_p,
        'MAE': mae,
        'RMSE': rmse,
        'QWK': qwk,
    }


def compute_metrics_for_trainer(eval_pred):
    """Compute metrics during training (for Trainer callback)."""
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    metrics = compute_metrics_from_predictions(labels, predictions)
    return {
        'pearson_r': metrics['Pearson r'],
        'spearman_r': metrics['Spearman ρ'],
        'mae': metrics['MAE'],
        'rmse': metrics['RMSE'],
        'qwk': metrics['QWK'],
    }


# ============================================================
# DATA LOADING
# ============================================================

def load_combined_splits(splits_dir: Path):
    """
    Load and combine wo_chat + withchat splits into single train/dev/test.
    """
    splits = {}
    for split_name in ['train', 'dev', 'test']:
        dfs = []
        for prefix in ['wo_chat', 'withchat']:
            path = splits_dir / f"{prefix}_{split_name}.csv"
            if path.exists():
                df = pd.read_csv(path)
                dfs.append(df)
                print(f"  Loaded {path.name}: {len(df)} rows")
            else:
                print(f"  Warning: {path} not found")

        if dfs:
            combined = pd.concat(dfs, ignore_index=True)
            combined = combined.dropna(subset=[SCORE_COL])
            splits[split_name] = combined
            print(f"  Combined {split_name}: {len(combined)} rows")
        else:
            print(f"  ERROR: No data for {split_name}")
            sys.exit(1)

    return splits


## Model: Feature injection via classifier-head replacementThe trick: load a normal `AutoModelForSequenceClassification` (which handles all thegradient plumbing, weight init, loss computation, etc.), then **swap out** only its`classifier` head with a small MLP that takes `[CLS] ⊕ features`.A tiny custom Trainer override passes `keystroke_features` through — everything else(gradient scaling, checkpointing, metric logging) stays stock HF.

In [6]:
# ============================================================
# MODEL: Feature injection via forward override
# ============================================================

from transformers.modeling_outputs import SequenceClassifierOutput


def build_model(model_name, num_features, hidden_size=128, dropout=0.1):
    """
    Load standard AutoModelForSequenceClassification, then:
      1. Replace classifier head with fusion MLP
      2. Override forward on the instance to accept keystroke_features
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression",
    )

    pooler_dim = model.pooler.output_dim  # 768

    # ── Replace classifier with fusion MLP ──
    model.classifier = nn.Sequential(
        nn.Linear(pooler_dim + num_features, hidden_size),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_size, 1),
    )
    nn.init.xavier_normal_(model.classifier[0].weight)
    nn.init.zeros_(model.classifier[0].bias)
    nn.init.xavier_normal_(model.classifier[3].weight)
    nn.init.zeros_(model.classifier[3].bias)

    # ── Store num_features for the forward override ──
    model._num_side_features = num_features

    # ── Override forward on this instance ──
    import types

    def forward_with_features(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        keystroke_features=None,
        **kwargs,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.deberta(
            input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        encoder_layer = outputs[0]
        pooled_output = self.pooler(encoder_layer)
        pooled_output = self.dropout(pooled_output)

        # Force float32 for the MLP
        pooled_output = pooled_output.float()

        if keystroke_features is not None:
            combined = torch.cat([pooled_output, keystroke_features.float()], dim=1)
        else:
            zeros = torch.zeros(
                pooled_output.size(0), self._num_side_features,
                device=pooled_output.device, dtype=torch.float32,
            )
            combined = torch.cat([pooled_output, zeros], dim=1)

        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(logits.squeeze(), labels.squeeze())

        if not return_dict:
            output = (logits,) + outputs[1:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    model.forward = types.MethodType(forward_with_features, model)
    return model


# ============================================================
# CUSTOM TRAINER (minimal — just passes keystroke_features)
# ============================================================

class FeatureInjectionTrainer(Trainer):
    """Pass keystroke_features to model.forward(). That's it."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.get("loss") if isinstance(outputs, dict) else outputs[0]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.get("loss") if isinstance(outputs, dict) else outputs[0]
            logits = outputs.get("logits") if isinstance(outputs, dict) else outputs[1]
        labels = inputs.get("labels")
        return (loss, logits, labels)

In [7]:
# ============================================================
# MAIN
# ============================================================

def main():
    parser = argparse.ArgumentParser(description="DeBERTa + keystroke features essay scoring")
    parser.add_argument('--eval_only', action='store_true')
    parser.add_argument('--checkpoint', type=str, default=None)
    parser.add_argument('--model_name', type=str, default=MODEL_NAME)
    parser.add_argument('--max_length', type=int, default=MAX_LENGTH)
    parser.add_argument('--epochs', type=int, default=NUM_EPOCHS)
    parser.add_argument('--batch_size', type=int, default=BATCH_SIZE)
    parser.add_argument('--lr', type=float, default=LEARNING_RATE)
    parser.add_argument('--seed', type=int, default=SEED)
    args, _ = parser.parse_known_args()

    print("=" * 70)
    print("DeBERTa + Keystroke Feature Injection — Essay Scoring")
    print("=" * 70)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device: {device}")
    if device == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # ── Load text lookup ──────────────────────────────────────
    print(f"\nLoading text lookup from {TEXT_LOOKUP}...")
    if not TEXT_LOOKUP.exists():
        print(f"ERROR: {TEXT_LOOKUP} not found.")
        sys.exit(1)
    text_lookup = load_text_lookup(TEXT_LOOKUP)
    print(f"  Loaded {len(text_lookup)} texts")

    # ── Load splits ───────────────────────────────────────────
    print(f"\nLoading splits from {SPLITS_DIR}...")
    splits = load_combined_splits(SPLITS_DIR)

    # Verify feature columns
    sample_df = splits['train']
    feature_cols = [c for c in FEATURE_COLS if c in sample_df.columns]
    missing_features = [c for c in FEATURE_COLS if c not in sample_df.columns]
    if missing_features:
        print(f"\nWARNING: Missing feature columns: {missing_features}")
    print(f"Using {len(feature_cols)} keystroke features + {len(TIMEPOINTS)} timepoint indicators")

    # Fill NaN in feature columns
    for split_name in splits:
        splits[split_name][feature_cols] = splits[split_name][feature_cols].fillna(0.0)

    # ── Tokenizer ─────────────────────────────────────────────
    print(f"\nLoading tokenizer: {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)

    # ── Datasets ──────────────────────────────────────────────
    print("\nCreating datasets...")
    train_dataset = EssayWithFeaturesDataset(
        splits['train'], text_lookup, tokenizer, args.max_length,
        feature_cols, TIMEPOINTS)
    dev_dataset = EssayWithFeaturesDataset(
        splits['dev'], text_lookup, tokenizer, args.max_length,
        feature_cols, TIMEPOINTS)
    test_dataset = EssayWithFeaturesDataset(
        splits['test'], text_lookup, tokenizer, args.max_length,
        feature_cols, TIMEPOINTS)

    print(f"  Train: {len(train_dataset)}, Dev: {len(dev_dataset)}, Test: {len(test_dataset)}")

    # Total features = keystroke + timepoint one-hot
    num_features = train_dataset.num_features

    # ── Model ─────────────────────────────────────────────────
    print(f"\nBuilding model: DeBERTa + {num_features} features "
          f"({len(feature_cols)} keystroke + {len(TIMEPOINTS)} timepoint)")
    model = build_model(
        args.model_name,
        num_features=num_features,
        hidden_size=FEATURE_MLP_HIDDEN,
        dropout=FEATURE_DROPOUT,
    )
    model = model.float()  # force full fp32 — DeBERTa-v3 is unstable otherwise
    # ── Diagnostic: check feature distributions ──
    import numpy as np

    print("=== FEATURE DIAGNOSTICS ===")
    feats = train_dataset.features
    print(f"Shape: {feats.shape}")
    print(f"Any NaN:  {np.isnan(feats).any()}")
    print(f"Any Inf:  {np.isinf(feats).any()}")
    print(f"Min:      {feats.min():.4f}")
    print(f"Max:      {feats.max():.4f}")
    print(f"Mean:     {feats.mean():.4f}")
    print(f"Std:      {feats.std():.4f}")

    # Per-column stats
    print(f"\nPer-feature min/max:")
    for i, col in enumerate(feature_cols + [f"tp_{t}" for t in TIMEPOINTS]):
        col_data = feats[:, i]
        n_nan = np.isnan(col_data).sum()
        print(f"  {col:30s}  min={np.nanmin(col_data):8.3f}  max={np.nanmax(col_data):8.3f}  nan={n_nan}")

    # Check a single batch through the model
    print("\n=== SINGLE BATCH TEST ===")
    from torch.utils.data import DataLoader
    loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
    batch = next(iter(loader))
    batch = {k: v.to('cuda') for k, v in batch.items()}

    model.eval()
    model.cuda()
    with torch.no_grad():
        out = model(**batch)
        print(f"Loss:   {out.loss.item():.6f}")
        print(f"Logits: {out.logits.squeeze().tolist()}")
        print(f"Any NaN in logits: {torch.isnan(out.logits).any().item()}")
    model.train()



    # ── Training ──────────────────────────────────────────────
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_output = OUTPUT_DIR / "model"

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "checkpoints"),

        # Training
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=args.lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        seed=args.seed,
        max_grad_norm=1.0,

        # Evaluation & saving
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="pearson_r",
        greater_is_better=True,
        save_total_limit=2,

        # Logging
        logging_dir=str(OUTPUT_DIR / "logs"),
        logging_steps=50,
        report_to="none",

        # fp16=False — DeBERTa-v3 is more stable in fp32
        fp16=False,
        bf16=False,
        dataloader_num_workers=2,
    )

    trainer = FeatureInjectionTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics_for_trainer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    if not args.eval_only:
        print("\n" + "─" * 70)
        print("TRAINING")
        print("─" * 70)

        train_result = trainer.train()

        # Save best model
        model_output.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), model_output / "model.pt")
        tokenizer.save_pretrained(str(model_output))

        # Save feature config for reproducibility
        with open(model_output / "feature_config.json", 'w') as f:
            json.dump({
                "model_name": args.model_name,
                "feature_cols": feature_cols,
                "timepoints": TIMEPOINTS,
                "hidden_size": FEATURE_MLP_HIDDEN,
                "dropout": FEATURE_DROPOUT,
            }, f, indent=2)

        print(f"\nModel saved to: {model_output}")

        # Save training log
        log_history = pd.DataFrame(trainer.state.log_history)
        log_history.to_csv(OUTPUT_DIR / "training_log.csv", index=False)

    # ── Evaluation ────────────────────────────────────────────
    print("\n" + "─" * 70)
    print("EVALUATION")
    print("─" * 70)

    for split_name, dataset in [('dev', dev_dataset), ('test', test_dataset)]:
        print(f"\n  --- {split_name.upper()} SET ---")

        predictions = trainer.predict(dataset)
        y_pred = predictions.predictions.squeeze()
        y_true = np.array(dataset.scores)

        metrics = compute_metrics_from_predictions(y_true, y_pred)

        print(f"  n = {len(y_true)}")
        print(f"  Pearson r:  {metrics['Pearson r']:.4f}  (p={metrics['Pearson p']:.2e})")
        print(f"  Spearman ρ: {metrics['Spearman ρ']:.4f}  (p={metrics['Spearman p']:.2e})")
        print(f"  MAE:        {metrics['MAE']:.4f}")
        print(f"  RMSE:       {metrics['RMSE']:.4f}")
        print(f"  QWK:        {metrics['QWK']:.4f}")

        # Save predictions
        pred_df = pd.DataFrame({
            'filename': dataset.filenames,
            'condition': dataset.chat_used,
            'source_timepoint': dataset.timepoints,
            'true_score': y_true,
            'predicted_score': y_pred,
            'predicted_binned': bin_to_half(np.clip(y_pred, 0.0, 5.0)),
        })
        pred_path = OUTPUT_DIR / f"predictions_{split_name}.csv"
        pred_df.to_csv(pred_path, index=False)
        print(f"  Predictions saved: {pred_path.name}")

    # ── Save experiment config ────────────────────────────────
    config = {
        'model_name': args.model_name,
        'max_length': args.max_length,
        'learning_rate': args.lr,
        'batch_size': args.batch_size,
        'num_epochs': args.epochs,
        'warmup_ratio': WARMUP_RATIO,
        'weight_decay': WEIGHT_DECAY,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'seed': args.seed,
        'n_keystroke_features': len(feature_cols),
        'n_timepoint_features': len(TIMEPOINTS),
        'n_total_features': num_features,
        'feature_cols': feature_cols,
        'timepoints': TIMEPOINTS,
        'feature_mlp_hidden': FEATURE_MLP_HIDDEN,
        'feature_dropout': FEATURE_DROPOUT,
        'n_train': len(train_dataset),
        'n_dev': len(dev_dataset),
        'n_test': len(test_dataset),
        'condition': 'combined',
        'timestamp': datetime.now().isoformat(),
    }
    with open(OUTPUT_DIR / "config.json", 'w') as f:
        json.dump(config, f, indent=2)

    print(f"\nAll outputs saved to: {OUTPUT_DIR.absolute()}")


if __name__ == '__main__':
    main()


DeBERTa + Keystroke Feature Injection — Essay Scoring
Device: cuda
GPU: Tesla T4

Loading text lookup from BEA Paper/text_mega_lookup.tsv...
  Loaded 24546 texts

Loading splits from BEA Paper/combined...
  Loaded wo_chat_train.csv: 3058 rows
  Loaded withchat_train.csv: 3141 rows
  Combined train: 6199 rows
  Loaded wo_chat_dev.csv: 1020 rows
  Loaded withchat_dev.csv: 1056 rows
  Combined dev: 2076 rows
  Loaded wo_chat_test.csv: 1272 rows
  Loaded withchat_test.csv: 1302 rows
  Combined test: 2574 rows
Using 26 keystroke features + 6 timepoint indicators

Loading tokenizer: microsoft/deberta-v3-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Model saved to: BEA Paper/models_transformer/text_plus_features/model

──────────────────────────────────────────────────────────────────────
EVALUATION
──────────────────────────────────────────────────────────────────────

  --- DEV SET ---


  n = 2076
  Pearson r:  0.8209  (p=0.00e+00)
  Spearman ρ: 0.8274  (p=0.00e+00)
  MAE:        0.5442
  RMSE:       0.8210
  QWK:        0.8082
  Predictions saved: predictions_dev.csv

  --- TEST SET ---


  n = 2574
  Pearson r:  0.7878  (p=0.00e+00)
  Spearman ρ: 0.7943  (p=0.00e+00)
  MAE:        0.5852
  RMSE:       0.8970
  QWK:        0.7730
  Predictions saved: predictions_test.csv

All outputs saved to: /content/BEA Paper/models_transformer/text_plus_features
